# Customer Churn Prediction and Probability Calibration

**Research question:** How do Logistic Regression and Random Forest compare in predicting customer churn, and how reliable are their predicted probabilities?

This notebook documents a reproducible machine learning experiment on tabular customer data. The workflow covers data inspection, preprocessing, model selection, threshold analysis, probability calibration and held-out evaluation.

**Dataset:** IBM Telco Customer Churn, [`Telco-Customer-Churn.csv`](https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv). The CSV's reuse terms must be verified separately from the repository software license, and the access date recorded before submission.


## 1. Environment setup and data acquisition

The experiment runs in a Google Colab CPU environment. Output directories are created for figures and result files, software versions are recorded, and the CSV is retrieved from IBM's GitHub repository. The execution environment should be recorded alongside the final results to support reproducibility.


In [ ]:
from pathlib import Path
import sys, platform, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

Path("figures").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
START_TIME = time.time()

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
print("Python:", sys.version.split()[0], "scikit-learn:", sklearn.__version__)
print("Platform:", platform.platform())
display(df.head())
assert df.shape == (7043, 21), "Unexpected dataset shape: check source/version"
assert df["customerID"].is_unique, "Duplicate customer IDs: investigate before splitting"
assert set(df["Churn"].dropna().unique()) == {"Yes", "No"}


## 2. Exploratory data analysis

The dataset is inspected for its dimensions, feature types, missing values, duplicate identifiers and target-class distribution. The `TotalCharges` field is converted to numeric format because blank entries may otherwise be interpreted as text. Missing numeric values are imputed later within the training pipeline to avoid preprocessing leakage.


In [ ]:

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate customer IDs:")
print(df["customerID"].duplicated().sum())


In [ ]:
print(df["Churn"].value_counts())

print("\nChurn percentages:")
print(df["Churn"].value_counts(normalize=True) * 100)


In [ ]:
df["Churn"].value_counts().reindex(["No", "Yes"]).plot(kind="bar", color=["steelblue", "orange"])
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/churn_distribution.png", dpi=200)
plt.show()

## 3. Feature preparation and dataset partitioning

The `customerID` identifier is excluded from the predictors, and the binary target `Churn` is encoded as 0 (No) and 1 (Yes). A stratified split assigns 60% of records to training, 20% to validation and 20% to the held-out test set. Model and threshold selection are conducted without consulting test labels.


In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("Missing values after conversion:")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:

X = df.drop(columns=["Churn", "customerID"])

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:

from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])


In [ ]:
for label, yy in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(label, "n=", len(yy), "churn rate=", round(yy.mean(), 4))

## 4. Baseline and initial model training

A prevalence-based Dummy Classifier establishes a simple reference performance. Logistic Regression and Random Forest provide two contrasting machine learning approaches. Numeric features are imputed and scaled; categorical features are imputed and one-hot encoded within fitted pipelines. Initial validation performance is measured using Average Precision (AP), recall, precision and Brier score. The initial classification threshold is 0.50.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Dummy": DummyClassifier(strategy="prior"),
    "Logistic Regression": LogisticRegression(
        max_iter=2000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    )
}

trained_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    print(name, "trained successfully!")


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss
)

results = []

for name, model in trained_models.items():
    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "Model": name,
        "PR AUC (AP)": average_precision_score(
            y_val, probabilities
        ),
        "Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Brier Score": brier_score_loss(
            y_val, probabilities
        )
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))


In [ ]:

from sklearn.calibration import calibration_curve

plt.figure(figsize=(7, 6))

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="black",
    label="Perfect Calibration"
)

for name, model in trained_models.items():
    probabilities = model.predict_proba(X_val)[:, 1]

    fraction_positive, mean_predicted = calibration_curve(
        y_val,
        probabilities,
        n_bins=10,
        strategy="quantile"
    )

    plt.plot(
        mean_predicted,
        fraction_positive,
        marker="o",
        label=name
    )

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Actual Fraction of Churners")
plt.title("Validation Probability Calibration")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("figures/calibration_plot.png", dpi=200)
plt.show()


In [ ]:
results_df.to_csv("results/initial_validation_results.csv", index=False)

## 5. Hyperparameter selection

A limited, predefined set of hyperparameter combinations is compared on the validation set. The highest validation Average Precision determines the selected configuration for each model family. This procedure is a modest search rather than an exhaustive optimization. Repeated use of the validation set for subsequent choices is acknowledged as a methodological limitation.


In [ ]:

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
import pandas as pd

candidates = {
    "Logistic Regression": [
        LogisticRegression(
            C=c, max_iter=2000, random_state=42
        )
        for c in [0.01, 0.1, 1, 10]
    ],
    "Random Forest": [
        RandomForestClassifier(
            n_estimators=200,
            max_depth=depth,
            min_samples_leaf=leaf,
            random_state=42,
            n_jobs=-1
        )
        for depth in [5, 10, None]
        for leaf in [2, 5]
    ]
}

best_models = {}
tuning_results = []

for name, model_list in candidates.items():
    best_score = -1

    for model in model_list:
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", model)
        ])

        pipeline.fit(X_train, y_train)

        probabilities = pipeline.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, probabilities)

        tuning_results.append({
            "Model": name,
            "Settings": str(model.get_params()),
            "Validation AP": score
        })

        if score > best_score:
            best_score = score
            best_models[name] = pipeline

    print(name, "Best validation AP:", round(best_score, 4))

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df[["Model", "Validation AP"]].round(4))


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss
)

tuned_results = []

for name, model in best_models.items():
    prob = model.predict_proba(X_val)[:, 1]
    pred = (prob >= 0.5).astype(int)

    tuned_results.append({
        "Model": name,
        "PR AUC (AP)": average_precision_score(y_val, prob),
        "Recall": recall_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Brier Score": brier_score_loss(y_val, prob)
    })

tuned_df = pd.DataFrame(tuned_results)

display(tuned_df.round(4))


In [ ]:
for name, pipeline in best_models.items():
    model = pipeline.named_steps["classifier"]
    if name == "Logistic Regression":
        print(name, "C=", model.C, "max_iter=", model.max_iter)
    else:
        print(name, "n_estimators=", model.n_estimators,
              "max_depth=", model.max_depth, "min_samples_leaf=", model.min_samples_leaf)

## 6. Classification-threshold analysis

Recall and precision are calculated at multiple decision thresholds to examine the trade-off between missed churners and false-positive alerts. For this experiment, the operating threshold is selected to maximize validation recall subject to a minimum validation precision of 40%. This 40% requirement is an illustrative modeling assumption, not an externally established business constraint.


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import recall_score, precision_score

threshold_results = []

for name, model in best_models.items():
    probabilities = model.predict_proba(X_val)[:, 1]

    for threshold in [0.2, 0.3, 0.4, 0.5, 0.6]:
        predictions = (probabilities >= threshold).astype(int)

        threshold_results.append({
            "Model": name,
            "Threshold": threshold,
            "Recall": recall_score(
                y_val, predictions, zero_division=0
            ),
            "Precision": precision_score(
                y_val, predictions, zero_division=0
            )
        })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))


In [ ]:

plt.figure(figsize=(8, 5))

for name, group in threshold_df.groupby("Model"):
    plt.plot(
        group["Threshold"],
        group["Recall"],
        marker="o",
        label=name
    )

plt.xlabel("Classification Threshold")
plt.ylabel("Recall")
plt.title("Recall at Different Classification Thresholds")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("figures/threshold_recall.png", dpi=200)
plt.show()


In [ ]:

import numpy as np

selected_thresholds = {}

for name, model in best_models.items():
    probabilities = model.predict_proba(X_val)[:, 1]

    choices = []

    for threshold in np.arange(0.05, 0.951, 0.01):
        predictions = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_val, predictions, zero_division=0
        )
        recall = recall_score(
            y_val, predictions, zero_division=0
        )

        if precision >= 0.40:
            choices.append((recall, threshold))

    if choices:
        best_recall = max(item[0] for item in choices)
        selected_thresholds[name] = max(
            threshold for recall, threshold in choices
            if recall == best_recall
        )
    else:
        selected_thresholds[name] = 0.5

print("Selected thresholds:")
print(selected_thresholds)


In [ ]:

from sklearn.metrics import precision_score, recall_score

for name, model in best_models.items():
    threshold = selected_thresholds[name]
    probabilities = model.predict_proba(X_val)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    print(f"\n{name}")
    print(f"Threshold: {threshold:.2f}")
    print(f"Recall: {recall_score(y_val, predictions):.4f}")
    print(f"Precision: {precision_score(y_val, predictions):.4f}")


## 7. Probability calibration assessment

Sigmoid calibration is fitted using five-fold cross-validation within the training data. Original and calibrated probability estimates are compared on the validation set using Average Precision, Brier score and reliability diagrams. A lower Brier score indicates lower mean squared probability error; visual closeness to the diagonal does not establish perfect calibration.


In [ ]:

from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone

calibrated_models = {}

for name, model in best_models.items():

    calibrated = CalibratedClassifierCV(
        estimator=clone(model),
        method="sigmoid",
        cv=5
    )

    calibrated.fit(X_train, y_train)

    calibrated_models[name] = calibrated

    print(name, "calibrated successfully!")


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss
)
import pandas as pd

calibration_results = []

for name in best_models:

    for version, model in [
        ("Original", best_models[name]),
        ("Calibrated", calibrated_models[name])
    ]:

        probabilities = model.predict_proba(X_val)[:, 1]

        calibration_results.append({
            "Model": name,
            "Version": version,
            "PR AUC (AP)": average_precision_score(
                y_val, probabilities
            ),
            "Brier Score": brier_score_loss(
                y_val, probabilities
            )
        })

calibration_df = pd.DataFrame(calibration_results)

display(calibration_df.round(4))


In [ ]:

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name in zip(
    axes, ["Logistic Regression", "Random Forest"]
):

    ax.plot(
        [0, 1], [0, 1],
        "k--", label="Perfect Calibration"
    )

    for label, model in [
        ("Original", best_models[name]),
        ("Calibrated", calibrated_models[name])
    ]:

        probabilities = model.predict_proba(X_val)[:, 1]

        observed, predicted = calibration_curve(
            y_val,
            probabilities,
            n_bins=10,
            strategy="quantile"
        )

        ax.plot(
            predicted,
            observed,
            marker="o",
            label=label
        )

    ax.set_title(name)
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Observed Churn Rate")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("figures/calibration_comparison.png", dpi=200)
plt.show()


## 8. Final model specification

For each model family, the original or calibrated probability version is selected using validation Brier score. A classification threshold is then selected for that version using the validation precision constraint. Model versions and thresholds are fixed before held-out test evaluation.


In [ ]:

from sklearn.metrics import brier_score_loss

final_models = {}

for name in best_models:

    original = best_models[name]
    calibrated = calibrated_models[name]

    original_prob = original.predict_proba(X_val)[:, 1]
    calibrated_prob = calibrated.predict_proba(X_val)[:, 1]

    original_brier = brier_score_loss(y_val, original_prob)
    calibrated_brier = brier_score_loss(y_val, calibrated_prob)

    if calibrated_brier < original_brier:
        final_models[name] = calibrated
        print(name, "-> Calibrated")
    else:
        final_models[name] = original
        print(name, "-> Original")


In [ ]:

import numpy as np
from sklearn.metrics import precision_score, recall_score

final_thresholds = {}

for name, model in final_models.items():

    probabilities = model.predict_proba(X_val)[:, 1]
    candidates = []

    for threshold in np.arange(0.05, 0.951, 0.01):

        predictions = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_val, predictions, zero_division=0
        )
        recall = recall_score(
            y_val, predictions, zero_division=0
        )

        if precision >= 0.40:
            candidates.append((recall, threshold))

    if candidates:
        best_recall = max(item[0] for item in candidates)

        final_thresholds[name] = max(
            threshold for recall, threshold in candidates
            if recall == best_recall
        )
    else:
        raise ValueError(
            f"No threshold meets precision constraint for {name}"
        )

    threshold = final_thresholds[name]
    predictions = (probabilities >= threshold).astype(int)

    print(f"\n{name}")
    print(f"Threshold: {threshold:.2f}")
    print(f"Validation Recall: {recall_score(y_val, predictions):.4f}")
    print(f"Validation Precision: {precision_score(y_val, predictions):.4f}")


## 9. Held-out test evaluation

The finalized models are evaluated on the previously reserved test set. The reported outputs include a model comparison table, a precision–recall curve, confusion matrices and a probability calibration plot. The Dummy Classifier is included as a reference baseline. Test results are used for reporting, not further hyperparameter or threshold adjustment.


In [ ]:

import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss,
    confusion_matrix
)

test_results = []

for name, model in final_models.items():

    probabilities = model.predict_proba(X_test)[:, 1]
    threshold = final_thresholds[name]
    predictions = (probabilities >= threshold).astype(int)

    test_results.append({
        "Model": name,
        "Threshold": threshold,
        "PR AUC (AP)": average_precision_score(
            y_test, probabilities
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Brier Score": brier_score_loss(
            y_test, probabilities
        )
    })

    print(f"\n{name} Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

test_df = pd.DataFrame(test_results)
display(test_df.round(4))


In [ ]:
from sklearn.metrics import average_precision_score, recall_score, precision_score, brier_score_loss
baseline = trained_models["Dummy"]
base_prob = baseline.predict_proba(X_test)[:, 1]
base_pred = baseline.predict(X_test)
baseline_row = pd.DataFrame([{
    "Model": "Dummy", "Threshold": 0.5,
    "PR AUC (AP)": average_precision_score(y_test, base_prob),
    "Recall": recall_score(y_test, base_pred, zero_division=0),
    "Precision": precision_score(y_test, base_pred, zero_division=0),
    "Brier Score": brier_score_loss(y_test, base_prob)
}])
complete_test_df = pd.concat([baseline_row, test_df], ignore_index=True)
display(complete_test_df.round(4))
complete_test_df.to_csv("results/final_model_comparison.csv", index=False)

In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

plt.figure(figsize=(8, 6))

for name, model in final_models.items():

    probabilities = model.predict_proba(X_test)[:, 1]

    precision, recall, _ = precision_recall_curve(
        y_test, probabilities
    )

    ap = average_precision_score(y_test, probabilities)

    plt.plot(
        recall, precision,
        label=f"{name} (AP={ap:.3f})"
    )

plt.axhline(
    y_test.mean(),
    linestyle="--",
    color="gray",
    label="Churn prevalence"
)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Final Test Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/final_pr_curve.png", dpi=200)
plt.show()


In [ ]:

from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model) in zip(axes, final_models.items()):

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (
        probabilities >= final_thresholds[name]
    ).astype(int)

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        display_labels=["Stayed", "Churned"],
        ax=ax,
        colorbar=False
    )

    ax.set_title(name)

plt.tight_layout()
plt.savefig("figures/final_confusion_matrices.png", dpi=200)
plt.show()


In [ ]:

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.plot(
    [0, 1], [0, 1],
    "k--", label="Perfect Calibration"
)

for name, model in final_models.items():
    probabilities = model.predict_proba(X_test)[:, 1]

    observed, predicted = calibration_curve(
        y_test,
        probabilities,
        n_bins=10,
        strategy="quantile"
    )

    plt.plot(
        predicted, observed,
        marker="o",
        label=name
    )

plt.xlabel("Mean Predicted Churn Probability")
plt.ylabel("Observed Churn Rate")
plt.title("Final Test Probability Calibration")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/final_test_calibration.png", dpi=200)
plt.show()


## 10. Results export and reproducibility

Evaluation tables, figures and environment information are exported for inclusion in the repository and written report. The final documentation should specify the Colab runtime, CPU/RAM information, execution date, dependency versions, random seed and dataset access instructions.

In [ ]:
tuning_df.to_csv("results/hyperparameter_tuning.csv", index=False)
calibration_df.to_csv("results/calibration_comparison.csv", index=False)
metadata = {
    "python": sys.version.split()[0],
    "scikit_learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "platform": platform.platform(),
    "random_state": 42,
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test),
    "runtime_seconds": round(time.time() - START_TIME, 1),
    "thresholds": {name: float(value) for name, value in final_thresholds.items()},
    "selected_settings": {name: str(pipe.named_steps["classifier"]) for name, pipe in best_models.items()}
}
Path("results/experiment_info.json").write_text(json.dumps(metadata, indent=2))
print("Saved files:")
for folder in ("results", "figures"):
    for file in sorted(Path(folder).glob("*")):
        print(file)
